In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option('max_colwidth', 1000)

In [ ]:
# Obtained from 01_publication_classification
ROOT_DISAMBIGUATED_CLASSIFICATION_DIR = '../data/open_alex_matches/'
out_dir = Path(ROOT_DISAMBIGUATED_CLASSIFICATION_DIR)

In [ ]:
# the set of papers that were able to be classified
matches_df = pd.read_csv(ROOT_DISAMBIGUATED_CLASSIFICATION_DIR + '/matches.csv.gz', engine = 'python', compression="gzip")

# collection of all the papers
dois_set = set(matches_df['doi'])

# save the papers
with open(out_dir / "doi_set", "w") as f:
    json.dump(list(dois_set), f)


# Databricks

In [ ]:
# https://huggingface.co/sentence-transformers
%pip install -U sentence-transformers

import json
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import pyarrow.dataset as ds

In [ ]:
ani_snapshot = "20260526/"
df_ani = spark.read.parquet(f"/mnt/els/seccont-anicore-parsed-edc/{ani_snapshot}")

# get the set of papers that were able to be classified
with open("doi_set", "r") as f:
    dois_set = set(json.load(f))
dois = pd.DataFrame({"doi": list(dois_set)})

# find the full entries in scopus for our papers
df_matches = df_ani.merge(dois, on="doi", how="inner")
df_abstract_doi = df_matches[["doi", "abstract"]]


from pyspark.sql import functions as func
from pyspark.sql.types import ArrayType, DoubleType

# embed locally
@func.pandas_udf(returnType=ArrayType(DoubleType()))
def encode(x: pd.Series) -> pd.Series:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("Snowflake/snowflake-arctic-embed-s", device="cuda")
    return pd.Series(model.encode(x.fillna("").tolist(), batch_size=32, show_progress_bar=True).tolist())

df_abstract_doi = df_abstract_doi.withColumn("abs_embedding", encode("abstract"))
df_abstract_doi.repartition(20).write.mode('overwrite').parquet(f'{root}encoded/abstract_embeddings.parquet')